In [12]:
import os  # file handling
import re  # regex for text processing
import math


# stopwords list
STOPWORDS = set([
    "a","an","the","is","are","was","were","in","on","at","of","and","or","to"
])

def normalize(word):
    return word.lower()  # convert to lowercase


# MySet class
class MySet:
    def __init__(self):
        self.data = set()  # internal set

    def addElement(self, element):
        self.data.add(element)  # add element

    def union(self, otherSet):
        result = MySet()  # create new set
        result.data = self.data.union(otherSet.data)  # union op
        return result  # return result

    def intersection(self, otherSet):
        result = MySet()  # create new set
        result.data = self.data.intersection(otherSet.data)  # intersect op
        return result  # return result


# Position class
class Position:
    def __init__(self, page_entry, word_index):
        self.page_entry = page_entry  # store page
        self.word_index = word_index  # store index

    def getPageEntry(self):
        return self.page_entry  # return page

    def getWordIndex(self):
        return self.word_index  # return index


# WordEntry class
class WordEntry:
    def __init__(self, word):
        self.word = word  # store word
        self.positions = []  # list of positions

    def addPosition(self, position):
        self.positions.append(position)  # add position

    def addPositions(self, positions):
        self.positions.extend(positions)  # add multiple

    def getAllPositionsForThisWord(self):
        return self.positions  # return positions

    def getTermFrequency(self, word):
        return len(self.positions)  # count freq


# PageIndex class
class PageIndex:
    def __init__(self):
        self.word_map = {}  # map word -> WordEntry

    def addPositionForWord(self, word, position):
        word = normalize(word)  # normalize word

        if word not in self.word_map:
            self.word_map[word] = WordEntry(word)  # create entry

        self.word_map[word].addPosition(position)  # add position

    def getWordEntries(self):
        return list(self.word_map.values())  # return all entries

    def getWordEntry(self, word):
        return self.word_map.get(word)  # get specific word


# PageEntry class
class PageEntry:
    def __init__(self, page_name, text):
        self.page_name = page_name  # store page name
        self.page_index = PageIndex()  # create index
        self.process(text)  # process text

    def process(self, text):
        words = re.findall(r'\w+', text)  # extract words

        index = 0  # word index

        for w in words:
            w = normalize(w)  # normalize

            if w in STOPWORDS:
                continue  # skip stopwords

            pos = Position(self, index)  # create position
            self.page_index.addPositionForWord(w, pos)  # add to index
            index += 1  # increment index

    def getPageIndex(self):
        return self.page_index  # return index

    def getPageName(self):
        return self.page_name  # return name


# MyHashTable class
class MyHashTable:
    def __init__(self):
        self.table = {}  # internal map

    def getHashIndex(self, string):
        return hash(string) % 1000  # hash function

    def addPositionsForWord(self, word_entry):
        word = word_entry.word  # get word

        if word not in self.table:
            self.table[word] = WordEntry(word)  # create entry

        self.table[word].addPositions(word_entry.getAllPositionsForThisWord())  # merge

    def getWordEntry(self, word):
        return self.table.get(word)  # return entry


# InvertedPageIndex class
class InvertedPageIndex:
    def __init__(self):
        self.hash_table = MyHashTable()  # create table
        self.pages = {}  # store pages

    def addPage(self, page_entry):
        self.pages[page_entry.getPageName()] = page_entry  # store page

        for entry in page_entry.getPageIndex().getWordEntries():
            self.hash_table.addPositionsForWord(entry)  # add to index

    def getPagesWhichContainWord(self, word):
        result = MySet()  # result set

        entry = self.hash_table.getWordEntry(word)  # lookup
        if not entry:
            return result  # empty

        for pos in entry.getAllPositionsForThisWord():
            result.addElement(pos.getPageEntry())  # add page

        return result  # return result

    def getPage(self, page_name):
        return self.pages.get(page_name)  # get page


# SearchEngine class
class SearchEngine:
    def __init__(self, dataset_path):
        self.dataset_path = dataset_path  # store path
        self.index = InvertedPageIndex()  # create index

    def addPage(self, page_name):
        page_path = os.path.join(self.dataset_path, "webpages", page_name)  # path

        if not os.path.exists(page_path):
            print(f"No webpage {page_name} found")  # error
            return

        text = ""  # text buffer

        if os.path.isdir(page_path):  # folder case
            for file in os.listdir(page_path):

                if file.startswith("."):
                    continue  # ignore hidden

                file_path = os.path.join(page_path, file)

                if not os.path.isfile(file_path):
                    continue  # skip non-files

                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        text += f.read().lower() + " "  # read file
                except:
                    pass  # ignore errors

        else:  # single file
            with open(page_path, "r", encoding="utf-8") as f:
                text = f.read().lower()  # read content

        page_entry = PageEntry(page_name, text)  # create page
        self.index.addPage(page_entry)  # add to index

    def performAction(self, action):
        parts = action.strip().split()  # split command

        if parts[0] == "addPage":
            self.addPage(parts[1])  # add page

        elif parts[0] == "queryFindPagesWhichContainWord":
            word = normalize(parts[1])  # normalize

            pages = self.index.getPagesWhichContainWord(word)  # search

            if not pages.data:
                print(f"No webpage contains word {word}")  # no result
            else:
                names = sorted([p.getPageName() for p in pages.data])  # sort
                print(", ".join(names))  # print

        elif parts[0] == "queryFindPositionsOfWordInAPage":
            word = normalize(parts[1])  # normalize
            page_name = parts[2]  # page name

            page = self.index.getPage(page_name)  # get page

            if not page:
                print(f"No webpage {page_name} found")  # error
                return

            entry = page.getPageIndex().getWordEntry(word)  # get word

            if not entry:
                print(f"Webpage {page_name} does not contain word {word}")  # not found
                return

            positions = [str(p.getWordIndex()) for p in entry.getAllPositionsForThisWord()]  # positions
            print(", ".join(positions))  # print


# process actions
def process_actions(engine, actions_file):
    with open(actions_file, "r") as f:
        for line in f:
            if line.strip():
                engine.performAction(line)  # process each line


class SearchEngine:

    def __init__(self, dataset_path):
        self.dataset_path = os.path.abspath(dataset_path)
        self.webpages_path = os.path.join(self.dataset_path, "webpages")
        
        self.index = {}  # word -> {page: freq}
        self.pages = set()
        
        print(f"Using dataset_path: {self.dataset_path}")
        print(f"[INIT] dataset_path = {self.dataset_path}\n")

    def clean_text(self, text):
        text = text.lower()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        return text.split()

    def get_page_file(self, page_name):
        """
        Try multiple possible file formats
        """
        # Case 1: .txt file
        path1 = os.path.join(self.webpages_path, page_name + ".txt")
        if os.path.exists(path1):
            return path1
        
        # Case 2: file without extension
        path2 = os.path.join(self.webpages_path, page_name)
        if os.path.isfile(path2):
            return path2
        
        # Case 3: folder containing file
        path3 = os.path.join(self.webpages_path, page_name, page_name + ".txt")
        if os.path.exists(path3):
            return path3
        
        return None

    def add_page(self, page_name):
        print(f"[ADD PAGE] {page_name}")
        
        if page_name in self.pages:
            print("OUTPUT: Page already added\n")
            return
        
        file_path = self.get_page_file(page_name)
        
        if file_path is None:
            print("OUTPUT: Page not found\n")
            return
        
        print(f"[DEBUG] Found file at: {file_path}")
        
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()
        
        words = self.clean_text(text)
        
        word_count = {}
        for word in words:
            word_count[word] = word_count.get(word, 0) + 1
        
        for word, freq in word_count.items():
            if word not in self.index:
                self.index[word] = {}
            self.index[word][page_name] = freq
        
        self.pages.add(page_name)
        
        print("OUTPUT: Page added\n")

    def query_word(self, word):
        word = word.lower()
        
        if word not in self.index:
            print("OUTPUT: No webpage contains word\n")
            return
        
        pages = sorted(self.index[word].keys())
        print(f"OUTPUT: {', '.join(pages)}\n")

    def query_positions(self, word, page_name):
        file_path = self.get_page_file(page_name)
        
        if file_path is None:
            print("OUTPUT: Page not found\n")
            return
        
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()
        
        words = self.clean_text(text)
        
        positions = []
        for i, w in enumerate(words):
            if w == word.lower():
                positions.append(str(i + 1))
        
        if not positions:
            print("OUTPUT: Word not found in page\n")
        else:
            print(f"OUTPUT: {', '.join(positions)}\n")

    def print_tfidf_logs(query_words, pages, tf, df, total_pages, scores):
        print("\n========== TF-IDF DETAILS ==========")
    
        for word in query_words:
            if word not in df:
                print(f"\nWord: '{word}' → NOT FOUND in any page")
                continue
    
            idf = math.log(total_pages / df[word])
    
            print(f"\nWord: '{word}'")
            print(f"  IDF = log({total_pages}/{df[word]}) = {idf:.4f}")
            print("  Page-wise:")
    
            for page in pages:
                term_freq = tf[page].get(word, 0)
                score = term_freq * idf
    
                if term_freq > 0:
                    print(f"    {page:25} | TF = {term_freq:3} | Score = {score:.4f}")
    
        print("\n========== FINAL RANKING ==========")
    
        for page, score in scores:
            print(f"{page:25} | TOTAL SCORE = {score:.4f}")
    
        print("===================================\n")

    def query_tfidf(self, words):
        scores = {}
        N = len(self.pages)
    
        print("\n========== TF-IDF SEARCH ==========")
        print(f"Total Pages: {N}")
    
        for word in words:
            word = word.lower()
    
            if word not in self.index:
                print(f"\nWord: '{word}' → NOT FOUND")
                continue
    
            pages_dict = self.index[word]
            df = len(pages_dict)
            idf = math.log(N / df) if df else 0
    
            print(f"\nWord: '{word}'")
            print(f"  DF = {df} | IDF = log({N}/{df}) = {idf:.4f}")
            print("  Contributions:")
    
            for page, tf in pages_dict.items():
                score = tf * idf
                print(f"    {page:25} | TF = {tf:3} | TF-IDF = {score:.4f}")
    
                scores[page] = scores.get(page, 0) + score
    
        if not scores:
            print("\nOUTPUT: No webpage contains given query\n")
            return
    
        ranked = sorted(scores.items(), key=lambda x: (-x[1], x[0]))
    
        print("\n========== FINAL RANKING ==========")
        for p, s in ranked:
            print(f"{p:25} | TOTAL = {s:.4f}")
    
        result = [p for p, _ in ranked]
    
        print("===================================")
        print(f"OUTPUT: {', '.join(result)}\n")
   
    
    def performAction(self, action):
        action = action.strip()
        print(f"[QUERY] {action}")
        
        parts = action.split()
        
        if parts[0] == "addPage":
            self.add_page(parts[1])
        
        elif parts[0] == "queryFindPagesWhichContainWord":
            self.query_word(parts[1])
        
        elif parts[0] == "queryFindPositionsOfWordInAPage":
            self.query_positions(parts[1], parts[2])
        
        elif parts[0] == "queryFindPagesWhichContainAllWords":
            self.query_tfidf(parts[1:])
        
        else:
            print("OUTPUT: Invalid query\n")

In [13]:
def main():
    import os  # for path handling

    # absolute dataset path (change if needed)
    dataset_path = "/home/rajesh/CSL7100-Assignment4/Part2/dataset/Q2_websearch"
    
    # actions file path
    actions_file = os.path.join(dataset_path, "actions.txt")

    # initialize engine
    engine = SearchEngine(dataset_path)

    print("[INFO] Processing queries...\n")

    # check if actions file exists
    if not os.path.exists(actions_file):
        print(f"ERROR: actions.txt not found at {actions_file}")
        return

    # read and process each action
    with open(actions_file, 'r') as f:
        for line in f:
            line = line.strip()
            
            if not line:
                continue  # skip empty lines
            
            engine.performAction(line)


# run main
if __name__ == "__main__":
    main()

Using dataset_path: /home/rajesh/CSL7100-Assignment4/Part2/dataset/Q2_websearch
[INIT] dataset_path = /home/rajesh/CSL7100-Assignment4/Part2/dataset/Q2_websearch

[INFO] Processing queries...

[QUERY] addPage stack_datastructure_wiki
[ADD PAGE] stack_datastructure_wiki
[DEBUG] Found file at: /home/rajesh/CSL7100-Assignment4/Part2/dataset/Q2_websearch/webpages/stack_datastructure_wiki
OUTPUT: Page added

[QUERY] queryFindPagesWhichContainWord delhi
OUTPUT: No webpage contains word

[QUERY] queryFindPagesWhichContainWord stack
OUTPUT: stack_datastructure_wiki

[QUERY] queryFindPagesWhichContainWord wikipedia
OUTPUT: stack_datastructure_wiki

[QUERY] queryFindPositionsOfWordInAPage magazines stack_datastructure_wiki
OUTPUT: Word not found in page

[QUERY] queryFindPagesWhichContainWord allain
OUTPUT: No webpage contains word

[QUERY] addPage stack_cprogramming
[ADD PAGE] stack_cprogramming
[DEBUG] Found file at: /home/rajesh/CSL7100-Assignment4/Part2/dataset/Q2_websearch/webpages/stack_cp

AttributeError: 'SearchEngine' object has no attribute 'page_words'